# DADA2: 04 — Evaluation

Three TODOs.

You will conclude the assignment by running a RAG eval.

Three questions to answer with numbers:
- Is the decision model as good a router as the chat model? 
- Does the diagnosis agent beat the ladder, and at what cost?
- which parts of what you built were worth building?

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import time
from rag import config, decide, evaluate, retrieval, agent, ladder

## Read this first

This notebook is where the Gemini quota goes. The ladder agent spends about
five chat calls per question and yours about three, and the judge spends one
more on each answer. Fourteen questions through both agents is roughly 150
calls against a limit of 500 a day.

So, you will want to run it once, when notebooks 02 and 03 already pass their final cells. The comparison cells sleep between questions on purpose. Start them and leave them alone.

In [ ]:
specific = evaluate.load_evalset("specific")
comparative = evaluate.load_evalset("comparative")
items = specific + comparative

print(f"{len(specific)} specific + {len(comparative)} comparative = {len(items)} labeled questions")
print("reranker:       cohere")
print("chat model:    ", config.CHAT_MODEL)
print("decision model:", config.DECISION_MODEL)

## 1. Two routers

### TODO 4.1 — the decision model against the chat model (1 pt)

Every labeled question has a kind: the 14 in `evalset.json` are specific,
the 10 in `evalset_hard.json` are comparative. Ask both models which kind each
one is, and keep the time.

Write the chat-model version yourself. It is a `with_structured_output` call
with a `Literal` field, the same thing you did for `Filter` in notebook 02.
Use `decide.SCOPES` for the descriptions so both models are asked the same
question.

**Steps:**

1. Define a scoper that uses Scope to get structured output from chat model
3. In the loop below, the decision-model line is written. Add the same two lines for the chat model, keyed `"chat"`: time the call, add the seconds, add one to `right` when the choice matches `kind`.


In [ ]:
from typing import Literal
from pydantic import BaseModel, Field
from rag.util import retry

# TODO 4.1 — a scoper that uses Scope to get structured output from chat model
class Scope(BaseModel):
    """Which kind of question this is."""
    scope: Literal["specific", "comparative", "figure"] = Field(
        description="; ".join(f"{k}: {v}" for k, v in decide.SCOPES.items()))

def chat_scope(question):
    t = time.time()
    out = retry(scoper.invoke, "Question about SEC 10-K filings: " + question
                + "\nWhat does it ask for?")
    return out.scope, time.time() - t

scoper = #TODO 4.1
# your code here

In [ ]:
labeled = [(i["question"], "specific") for i in specific] + \
           [(i["question"], "comparative") for i in comparative]

right = {"decision": 0, "chat": 0}
secs = {"decision": 0.0, "chat": 0.0}
for q, kind in labeled:
    t = time.time(); choice, _ = decide.scope(q); secs["decision"] += time.time() - t
    right["decision"] += (choice == kind)
    t = time.time(); choice, _ = chat_scope(q); secs["chat"] += time.time() - t
    right["chat"] += (choice == kind)
    time.sleep(3)

n = len(labeled)
for name in ("decision", "chat"):
    print(f"{name:9} {right[name]}/{n} right, {secs[name]/n:.2f}s per call")

NOTE: Accuracy is close, and the chat model is probably not
slower. If you expected the decision model to win on both, write down what it does give you instead: a probability per option, and a call that does not count against the daily limit. Whether those are worth having is the question
the next section answers.

## 2. Two agents

### TODO 4.2 — the ladder against the diagnosis (1 pt)

`evaluate.compare_agents` runs each agent over the questions, has the chat
model judge every answer, and keeps the bill: chat calls, decision calls, and
seconds per question. Both agents expose `run(question)` returning the final
state, which is all it needs.

Fourteen questions is enough. Use eight specific and six comparative so both
kinds are in the mix.

**Steps:** nothing to write. Run the cell once, when both agents pass their final cells. If a 429 cuts it off, score the two agents in separate cells rather than restarting the pair.

This will take a while. Plan accordingly. 

Make sure your rate limit is reset or use a different provider (e.g., Groq, OpenRouter) instead of the Gemini API. 

Let it run and come back to see your results.


In [ ]:
# TODO 4.2
subset = specific[:8] + comparative[:6]

rows = evaluate.compare_agents({"ladder": ladder.run, "diagnosis": agent.run}, subset)

In [ ]:
# the judge's reasons, on anything either agent got wrong
_, notes = evaluate.score_agent(agent.run, subset[:4])
for note in notes:
    mark = "ok " if note["correct"] else "WRONG"
    print(f"[{mark}] {note['question'][:66]}  ({note['chat_calls']} chat, {note['decision_calls']} decision)")
    print(f"        {note['reason'][:150]}\n")

If the diagnosis agent got something wrong that the ladder got right, find
out which half failed. Run `decide.scope` and `decide.diagnose` on that
question by hand and look at the probabilities. A wrong route and a wrong
verdict are different bugs, and the fix for one is a threshold while the fix
for the other is a search strategy.

### TODO 4.3 — write it up (2 pts)

In the cell below, in your own words. Four or five short paragraphs.

**Conditions.** Reranker, chat model, decision model, k, chunk size. Enough
for someone to re-run you.

**The routers.** Which was more accurate, which was faster, and by how much.
Then say what the decision model is for, given those numbers.

**The agents.** Accuracy, chat calls per question, seconds per question, for
both. Name the question where they differed most and say why.

**One thing that did not help.** Something you built that the numbers say you
could remove. Say what the mechanism is, not that the model is better at it.

**What you would ship**, and what you would want to know about your users'
questions first.

Two things that cost marks: an improvement the table does not show, and a
difference explained by "the model is better" when the reason is in your
code.

**Steps:** write it as prose, four or five short paragraphs. The bold headings in the cell below are the order; replace the cell's text under each. Every number you quote should be in a table above.


---

### Your write-up

*(replace this cell)*

**Conditions.**

**The routers.**

**The agents.**

**What did not help.**

**What I would ship.**

**Copy your write-up to README.md.**